# Molecular Dynamics Simulation — Urease Active Site
**OpenMM | Google Colab | Shayan Asadi**

شبیه‌سازی دینامیک مولکولی active site آنزیم اوره‌آز *H. pylori* (PDB: 4H9M)

> **Data source:** Supporting analysis for:
> Asadi, S. et al. *Enhanced urease inhibitory activity of quercetin via conjugation with silver nanoparticles.* Scientific Reports 15, 11892 (2025). https://doi.org/10.1038/s41598-025-96684-2

---

## Pipeline
1. نصب کتابخانه‌ها
2. دانلود و آماده‌سازی پروتئین (PDBFixer)
3. استخراج active site (232 residue، ~3485 اتم)
4. ساخت سیستم (AMBER14 + TIP3P water، 20080 اتم)
5. Energy Minimization
6. Equilibration (NVT + NPT، هر کدام 50 ps)
7. Production Run (100 ps)
8. آنالیز RMSD و RMSF

**⚠️ قبل از شروع:** Runtime → Change runtime type → **T4 GPU**

## Step 1 — نصب کتابخانه‌ها

- **OpenMM:** موتور اصلی MD — فیزیک نیوتونی F=ma را برای هر اتم حساب می‌کند
- **PDBFixer:** فایل PDB را fix می‌کند — اتم‌های گمشده و hydrogen اضافه می‌کند
- **MDAnalysis:** trajectory را آنالیز می‌کند — RMSD و RMSF محاسبه می‌کند

In [ ]:
!pip install -q openmm pdbfixer mdanalysis
print('✓ Libraries installed')

## Step 2 — دانلود و Fix پروتئین

**PDBFixer چیکار می‌کند:**
- `findMissingResidues`: residueهای گمشده را پیدا می‌کند
- `removeHeterogens`: آب و لیگاندهای اضافه را حذف می‌کند
- `addMissingHydrogens(7.0)`: hydrogen در pH فیزیولوژیک (7.0) اضافه می‌کند

بدون این مرحله، simulation crash می‌کند چون اتم‌های ناقص clash ایجاد می‌کنند.

In [ ]:
import urllib.request
from pdbfixer import PDBFixer
from openmm.app import PDBFile

# دانلود 4H9M از RCSB
urllib.request.urlretrieve('https://files.rcsb.org/download/4H9M.pdb', '4H9M.pdb')
print('✓ Downloaded 4H9M.pdb')

# Fix پروتئین
fixer = PDBFixer(filename='4H9M.pdb')
fixer.findMissingResidues()
fixer.findNonstandardResidues()
fixer.replaceNonstandardResidues()
fixer.removeHeterogens(keepWater=False)
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)

with open('receptor_fixed.pdb', 'w') as f:
    PDBFile.writeFile(fixer.topology, fixer.positions, f)
print('✓ receptor_fixed.pdb ready')

## Step 3 — استخراج Active Site

کل urease بیش از 200,000 اتم دارد که برای Colab سنگین است.

**راه‌حل:** فقط residueهای 20 آنگستروم دور active site را نگه می‌داریم.

**Active site center** (محل اتصال Ni²⁺): x=18.78, y=-57.81, z=-24.15 آنگستروم

این کار سیستم را به ~20,000 اتم کاهش می‌دهد بدون از دست دادن اطلاعات مهم.

In [ ]:
import numpy as np
from openmm.app import PDBFile
from openmm.unit import nanometers

pdb = PDBFile('receptor_fixed.pdb')

# مختصات active site به nm
center = np.array([18.78, -57.81, -24.15]) / 10

# پیدا کردن residueهای نزدیک active site (2 nm = 20 آنگستروم)
positions = pdb.positions.value_in_unit(nanometers)
close_residues = set()

for atom, pos in zip(pdb.topology.atoms(), positions):
    dist = np.linalg.norm(np.array(pos) - center)
    if dist < 2.0:
        close_residues.add(atom.residue.index)

print(f'✓ Residues near active site: {len(close_residues)}')
print(f'  Estimated atoms: ~{len(close_residues) * 15}')

# ساخت topology جدید برای active site
from openmm.app.topology import Topology
new_top = Topology()
new_pos = []
chain_map = {}
res_map = {}

for atom, pos in zip(pdb.topology.atoms(), pdb.positions):
    if atom.residue.index not in close_residues:
        continue
    chain = atom.residue.chain
    if chain not in chain_map:
        chain_map[chain] = new_top.addChain(chain.id)
    res = atom.residue
    if res.index not in res_map:
        res_map[res.index] = new_top.addResidue(res.name, chain_map[chain])
    new_top.addAtom(atom.name, atom.element, res_map[res.index])
    new_pos.append(pos)

# ذخیره active site
pos_array = np.array([[v.x, v.y, v.z] for v in new_pos]) * nanometers
with open('active_site.pdb', 'w') as f:
    PDBFile.writeFile(new_top, pos_array, f)
print('✓ active_site.pdb saved')

## Step 4 — Fix Active Site و ساخت سیستم

بعد از برش، terminal residueها ناقص می‌شوند. PDBFixer دوباره آن‌ها را fix می‌کند.

**Force field:**
- **AMBER14:** برای پروتئین — پارامترهای bond، angle، dihedral
- **TIP3P:** مدل آب — هر مولکول 3 نقطه‌ای
- **150 mM NaCl:** شرایط فیزیولوژیک

**Padding 8 آنگستروم:** ضخامت لایه آب دور پروتئین

In [ ]:
from pdbfixer import PDBFixer
from openmm.app import *
from openmm import *
from openmm.unit import *

# Fix active site
fixer2 = PDBFixer(filename='active_site.pdb')
fixer2.findMissingResidues()
fixer2.findMissingAtoms()
fixer2.addMissingAtoms()
fixer2.addMissingHydrogens(7.0)

with open('active_site_fixed.pdb', 'w') as f:
    PDBFile.writeFile(fixer2.topology, fixer2.positions, f)
print('✓ active_site_fixed.pdb ready')

# ساخت سیستم با آب و یون
forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
modeller = Modeller(fixer2.topology, fixer2.positions)
modeller.addSolvent(forcefield, model='tip3p', padding=8*angstroms, ionicStrength=0.15*molar)

print(f'✓ Total atoms in system: {modeller.topology.getNumAtoms()}')
print('  (protein active site + TIP3P water + NaCl ions)')
print('✓ System built!')

## Step 5 — Energy Minimization

**چرا minimization؟**
وقتی آب به سیستم اضافه می‌شود، بعضی مولکول‌های آب خیلی نزدیک به پروتئین قرار می‌گیرند (clash).
اگر simulation با clash شروع شود، نیروهای بسیار بزرگ ایجاد می‌شود و سیستم منفجر می‌شود.

Minimization انرژی را minimize می‌کند تا:
- Energy before: مثبت بزرگ (بی‌ثباتی)
- Energy after: منفی بزرگ (پایداری)

**Integrator Langevin:** ترموستات — دما را در 300K نگه می‌دارد
- friction 1/ps: چقدر سریع به دما equilibrate می‌شود
- timestep 2 fs: هر step چقدر به جلو می‌رود

In [ ]:
# ساخت system و integrator
system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=PME,          # Particle Mesh Ewald برای electrostatics
    nonbondedCutoff=1.0*nanometers,  # cutoff فاصله برای non-bonded interactions
    constraints=HBonds             # bond های H ثابت — timestep بزرگتر ممکن می‌شود
)

integrator = LangevinMiddleIntegrator(
    300*kelvin,      # دمای هدف
    1/picosecond,    # friction coefficient
    0.002*picoseconds  # timestep = 2 fs
)

simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)

# انرژی قبل از minimization
state = simulation.context.getState(getEnergy=True)
print(f'Energy before minimization: {state.getPotentialEnergy():.1f}')

# Minimization
print('Minimizing... (چند دقیقه طول می‌کشد)')
simulation.minimizeEnergy(maxIterations=500)

# انرژی بعد از minimization
state = simulation.context.getState(getEnergy=True, getPositions=True)
print(f'Energy after minimization:  {state.getPotentialEnergy():.1f}')
print('✓ Minimization complete — system is stable')

## Step 6 — Equilibration

قبل از production run، باید سیستم را equilibrate کرد:

**NVT (50 ps) — Number, Volume, Temperature constant:**
- Volume ثابت است
- دما به 300K می‌رسد و تثبیت می‌شود
- velocity‌ها از Maxwell-Boltzmann distribution تولید می‌شوند

**NPT (50 ps) — Number, Pressure, Temperature constant:**
- فشار 1 bar ثابت است (شرایط اتمسفر)
- density تثبیت می‌شود
- MonteCarloBarostat برای کنترل فشار اضافه می‌شود

بدون equilibration، production run نتایج معنادار نمی‌دهد.

In [ ]:
# NVT Equilibration — دما را تثبیت می‌کند
print('NVT equilibration (50 ps)...')
simulation.context.setVelocitiesToTemperature(300*kelvin)
simulation.step(25000)  # 25000 steps × 2 fs = 50 ps
print('✓ NVT done')

# NPT Equilibration — فشار را تثبیت می‌کند
system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
simulation.context.reinitialize(preserveState=True)
print('NPT equilibration (50 ps)...')
simulation.step(25000)
print('✓ NPT done — system is ready for production!')

## Step 7 — Production Run (100 ps)

**100 ps = 50,000 steps × 2 fs**

در طول production run:
- هر 1000 step یک frame از trajectory ذخیره می‌شود (50 frames کل)
- هر 1000 step انرژی و دما log می‌شود
- trajectory در Google Drive ذخیره می‌شود

**DCDReporter:** trajectory را در فرمت DCD ذخیره می‌کند
**StateDataReporter:** انرژی، دما، پیشرفت را log می‌کند

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ذخیره checkpoint قبل از production
simulation.saveCheckpoint('/content/drive/MyDrive/urease_checkpoint.chk')
print('✓ Checkpoint saved to Google Drive')

import sys

# اضافه کردن reporters
simulation.reporters.append(
    DCDReporter('/content/drive/MyDrive/urease_traj.dcd', 1000)
)
simulation.reporters.append(
    StateDataReporter(
        sys.stdout, 1000,
        step=True,
        potentialEnergy=True,
        temperature=True,
        progress=True,
        totalSteps=50000
    )
)

print('Production run starting (100 ps)...')
print('Format: Progress%, Step, Potential Energy (kJ/mol), Temperature (K)')
print('-' * 70)
simulation.step(50000)
print('-' * 70)
print('✓ Production run complete!')

## Step 8 — آنالیز: RMSD

**RMSD (Root Mean Square Deviation):**
می‌سنجد ساختار در هر لحظه چقدر از ساختار اولیه فاصله گرفته.

$$RMSD(t) = \sqrt{\frac{1}{N}\sum_{i=1}^{N}|r_i(t) - r_i(0)|^2}$$

**تفسیر:**
- RMSD < 2 Å: سیستم پایدار است
- RMSD plateau می‌کند: simulation به equilibrium رسیده
- RMSD مدام بالا می‌رود: سیستم unstable است

ما **backbone** را آنالیز می‌کنیم (Cα، N، C، O) — نه side chainها

In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis.rms import RMSD
import matplotlib.pyplot as plt
import numpy as np

# ذخیره topology درست برای MDAnalysis
state = simulation.context.getState(getPositions=True)
with open('simulation_topology.pdb', 'w') as f:
    PDBFile.writeFile(simulation.topology, state.getPositions(), f)

# load trajectory
u = mda.Universe('simulation_topology.pdb', '/content/drive/MyDrive/urease_traj.dcd')
print(f'✓ Trajectory loaded: {len(u.trajectory)} frames, {len(u.atoms)} atoms')

# محاسبه RMSD
backbone = u.select_atoms('backbone')
R = RMSD(backbone, backbone, select='backbone')
R.run()

# رسم نمودار
fig, ax = plt.subplots(figsize=(9, 4))

frames = R.results.rmsd[:, 0]
time_ps = frames * 2 / 1000  # steps to ps

ax.plot(time_ps, R.results.rmsd[:, 2], color='#1D9E75', linewidth=2)
ax.fill_between(time_ps, R.results.rmsd[:, 2], alpha=0.2, color='#1D9E75')
ax.set_xlabel('Time (ps)', fontsize=12)
ax.set_ylabel('RMSD (Å)', fontsize=12)
ax.set_title(
    'Backbone RMSD — Urease Active Site (PDB: 4H9M) | 100 ps MD\n'
    'Shayan Asadi et al., Scientific Reports 2025',
    fontsize=11
)
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)

mean_rmsd = np.mean(R.results.rmsd[10:, 2])  # میانگین بعد از equilibration
ax.axhline(mean_rmsd, color='red', linestyle='--', alpha=0.5, linewidth=1)
ax.text(time_ps[-1]*0.02, mean_rmsd+0.02, f'Mean: {mean_rmsd:.2f} Å', fontsize=9, color='red')

plt.tight_layout()
plt.savefig('rmsd_plot.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/rmsd_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ RMSD plot saved')

## Step 9 — آنالیز: RMSF

**RMSF (Root Mean Square Fluctuation):**
برای هر residue می‌سنجد در طول simulation چقدر از موقعیت میانگینش deviation داشته.

$$RMSF_i = \sqrt{\langle|r_i - \langle r_i \rangle|^2\rangle}$$

**تفسیر:**
- RMSF پایین: residue rigid است (معمولاً توی secondary structure)
- RMSF بالا: residue flexible است (معمولاً loop region)
- پیک‌های بالا در active site: ممکن است با ligand binding مرتبط باشد

ما از **Cα atoms** استفاده می‌کنیم — یک نقطه به ازای هر residue

In [ ]:
from MDAnalysis.analysis.rms import RMSF

# محاسبه RMSF
protein_ca = u.select_atoms('protein and name CA')
rmsf_r = RMSF(protein_ca).run()

# رسم نمودار
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(protein_ca.resids, rmsf_r.results.rmsf, color='#2196F3', linewidth=1.2)
ax.fill_between(protein_ca.resids, rmsf_r.results.rmsf, alpha=0.15, color='#2196F3')

# پیدا کردن flexible regions
threshold = np.mean(rmsf_r.results.rmsf) + np.std(rmsf_r.results.rmsf)
flexible = protein_ca.resids[rmsf_r.results.rmsf > threshold]
if len(flexible) > 0:
    ax.axhline(threshold, color='orange', linestyle='--', alpha=0.7, linewidth=1)
    ax.text(protein_ca.resids[0], threshold+0.02, 'Flexible threshold', fontsize=8, color='orange')

ax.set_xlabel('Residue Number', fontsize=12)
ax.set_ylabel('RMSF (Å)', fontsize=12)
ax.set_title(
    'Per-Residue RMSF — Urease Active Site (PDB: 4H9M) | 100 ps MD\n'
    'Shayan Asadi et al., Scientific Reports 2025',
    fontsize=11
)
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('rmsf_plot.png', dpi=150, bbox_inches='tight')
plt.savefig('/content/drive/MyDrive/rmsf_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ RMSF plot saved')
print(f'  Mean RMSF: {np.mean(rmsf_r.results.rmsf):.2f} Å')
print(f'  Flexible residues (>{threshold:.2f} Å): {list(flexible)}')

## Step 10 — دانلود نتایج

In [ ]:
from google.colab import files
import os

for f in ['rmsd_plot.png', 'rmsf_plot.png', 'simulation_topology.pdb']:
    if os.path.exists(f):
        files.download(f)
        print(f'✓ Downloaded: {f}')

print('\n✓ All files downloaded!')

---
## Methods Text (برای مقاله)

> Molecular dynamics simulation of the urease active site (PDB: 4H9M, residues within 20 Å of the Ni²⁺ catalytic center) was performed using OpenMM 8.5 with the AMBER14 force field and TIP3P water model. The system (20,080 atoms) was energy-minimized (500 iterations), followed by NVT (50 ps) and NPT (50 ps, 1 bar) equilibration at 300 K using a Langevin thermostat (friction 1 ps⁻¹) and Monte Carlo barostat. A 100 ps production run was performed with a 2 fs timestep. RMSD and RMSF were calculated using MDAnalysis.

---
**Author:** Shayan Asadi — Medicinal Chemistry, Zanjan University of Medical Sciences

**GitHub:** [@shayan-debug](https://github.com/shayan-debug)

**Paper DOI:** [10.1038/s41598-025-96684-2](https://doi.org/10.1038/s41598-025-96684-2)